In [5]:
import numpy as np
import pandas as pd

def keep_only_numerical_value(x):
    if isinstance(x, str):
        return ''.join(filter(str.isdigit, x))
    return x

dfo = pd.read_csv('src/1857_commercial_guide_20250220.csv')
dfp = pd.read_csv('src/1857_cleaned_olivier.csv')

dfp['PAGE_NUM'] = dfp['PAGE_NUM'].apply(keep_only_numerical_value)

page_first_occurence_of_name = {}

# had to do it this way because the join method from above made duplicates that were hard to deal with.
curr_page = 0
found_name = False
for i, row in dfp.iterrows():
    fn = row['FIRST_N']
    ln = row['LAST_N']
    if not found_name and not pd.isna(fn) and not pd.isna(ln):
        page_first_occurence_of_name[curr_page] = [fn, ln]
        found_name = True
    if not pd.isna(row['PAGE_NUM']):
        curr_page = row['PAGE_NUM']
        found_name = False
page_first_occurence_of_name
dfo['PAGE_NUM'] = np.nan
# manually fixing the page numbers based on the first occurrence of names
# so the sync process works correctly
page_first_occurence_of_name['156'] = ['Andrea', 'Carli']
page_first_occurence_of_name['275'] = ['Domenico,', 'Penso']
page_first_occurence_of_name['276'] = ['Eugenio,', 'Vio']

page_first_occurence_of_name
page_iter = list(page_first_occurence_of_name.items())
iter = 0
for idx, r in dfo.iterrows():
    fn = r['FIRST_N']
    ln = r['LAST_N']
    if fn == page_iter[iter][1][0] and ln == page_iter[iter][1][1]:
        print(f"Found {fn} {ln} at page {page_iter[iter][0]}")
        dfo.loc[idx, 'PAGE_NUM'] = page_iter[iter][0]
        iter += 1

# no match after the 315 (i.e. only 316) is fine as I checked manually, no points from the 316th page was georefenrenced yet.

# shift between the pdf pages number and the actual document page number
dfo['PAGE_NUM'] = dfo['PAGE_NUM'].ffill().apply(lambda v: int(v) + 6)
dfo.to_csv('src/1857_commercial_guide_with_pages_20250715.csv', index=False)

Found Antonio, Alessandri at page 103
Found Giovanni, Battistella at page 104
Found Antonio, Petris at page 105
Found Giovanni. Paulovich at page 138
Found Girolamo Levi at page 139
Found Bonizio Fortunatti at page 140
Found Marco Gariboldi at page 141
Found Giacomo; Pase; Luigi Carnielli at page 143
Found Sebastiano; Sante Padrun ;Sobrio at page 144
Found Gio.; Battista, Berlendis at page 145
Found Antonio, Daveglia at page 146
Found Catterina, Lovetti at page 147
Found Triaca; Maria, Sopelsa at page 148
Found Francesco, Galizzi at page 149
Found Tomaso, Fabruzzo at page 150
Found Francesco, Guidetti at page 151
Found Domenico Massaggia at page 152
Found Maria, Bettinelli at page 153
Found Giovanni, Battistin at page 154
Found Giuseppe, Abbiatici at page 155
Found Andrea Carli at page 156
Found Pietro, Dal Soldà at page 157
Found Rist; Giuseppina, Kosser at page 158
Found Maria, Pigotti at page 159
Found Luigi, Toso at page 160
Found Bartolammeo, Seibezzi at page 161
Found Margherita,

# offset from sampling of pages:
140 => pdf 147